In [1]:
"""
Dual-Stream YOLOv11n: Comb4 (NearIR+Signal+Reflect) backbone + Range side branch.

Use your EXISTING unmodified yolo11n.yaml — no yaml changes needed.

dual_stream.yaml:
    path:  .
    train: dataset_dual/images_comb4/train
    val:   dataset_dual/images_comb4/valid
    test:  dataset_dual/images_comb4/test
    nc:    1
    names: ["snowpole"]
"""

import cv2
import numpy as np
import torch
import torch.nn as nn
from copy import deepcopy
from pathlib import Path

from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules import Conv
from ultralytics.utils import LOGGER
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.data.dataset import YOLODataset

try:
    from ultralytics.utils.torch_utils import de_parallel
except ImportError:
    def de_parallel(model):
        return model.module if hasattr(model, "module") else model


# ════════════════════════════════════════════════════════════
# 1.  Range encoder
#     Input : (B, 1, H, W)
#     Output: (B, RANGE_CH, H/16, W/16)
# ════════════════════════════════════════════════════════════

class RangeEncoder(nn.Module):
    def __init__(self, out_ch: int = 128):
        super().__init__()
        self.encode = nn.Sequential(
            Conv(1,      16,     3, 2),   # → P1  /2
            Conv(16,     32,     3, 2),   # → P2  /4
            Conv(32,     64,     3, 2),   # → P3  /8
            Conv(64,     out_ch, 3, 2),   # → P4  /16
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encode(x)


# ════════════════════════════════════════════════════════════
# 2.  Dual-stream model
# ════════════════════════════════════════════════════════════

class DualStreamModel(DetectionModel):
    """
    Standard YOLOv11n (parsed from yaml as-is, ch=3) plus a lightweight
    range encoder injected at neck P4 (after layer INJECT_AFTER).

    Layer indices for yolo11n (0-indexed, backbone 0-10, head 11+):
      13 = C3k2 after P4 Concat  ← inject range here
    """

    INJECT_AFTER = 13
    RANGE_CH     = 128

    def __init__(self, cfg="yolo11n.yaml", ch=3, nc=None, verbose=True):
        # Build standard 3-ch model from yaml
        super().__init__(cfg=cfg, ch=ch, nc=nc, verbose=verbose)

        # Range branch — attached AFTER super().__init__() completes
        self.range_encoder = RangeEncoder(out_ch=self.RANGE_CH)
        p4_ch = self._p4_channels()
        self.range_proj = nn.Sequential(
            nn.Conv2d(p4_ch + self.RANGE_CH, p4_ch, 1, bias=False),
            nn.BatchNorm2d(p4_ch),
            nn.SiLU(),
        )
        LOGGER.info(
            f"DualStreamModel: range injected after layer {self.INJECT_AFTER}, "
            f"P4 ch={p4_ch}, range_ch={self.RANGE_CH}"
        )

    def _p4_channels(self) -> int:
        m = self.model[self.INJECT_AFTER]
        try:
            return m.cv2.conv.out_channels
        except AttributeError:
            return 128  # yolo11n-n width=0.25 → 512*0.25=128

    def _run_once(self, x_comb4: torch.Tensor, range_feat=None) -> torch.Tensor:
        """Layer-by-layer forward. Injects range_feat after INJECT_AFTER."""
        y   = []
        inp = x_comb4

        for i, m in enumerate(self.model):
            if m.f != -1:
                inp = (
                    y[m.f]
                    if isinstance(m.f, int)
                    else [inp if j == -1 else y[j] for j in m.f]
                )
            inp = m(inp)

            if i == self.INJECT_AFTER and range_feat is not None:
                inp = self.range_proj(
                    torch.cat([inp, range_feat], dim=1)
                )

            y.append(inp if m.i in self.save else None)

        return inp

    def _extract_and_forward(self, x: torch.Tensor) -> torch.Tensor:
        """Split 4-ch tensor, run both streams."""
        if not hasattr(self, "_range_debug_done"):
            print("\n[MODEL DEBUG]")
            print("Input shape:", x.shape)
            print("Range channel stats:",
                x[:, 3].min().item(),
                x[:, 3].max().item())
            self._range_debug_done = True
        x_comb4 = x[:, :3]
        # If input has 4 channels use the range channel, else use zeros
        # (3-ch case happens during validator warmup)
        if x.shape[1] >= 4:
            x_range = x[:, 3:4]
        else:
            x_range = torch.zeros(
                x.shape[0], 1, x.shape[2], x.shape[3],
                dtype=x.dtype, device=x.device
            )
        range_feat = self.range_encoder(x_range)
        return self._run_once(x_comb4, range_feat)

    # ── forward: handles tensor (inference) AND dict (training via PyTorch __call__)
    def forward(self, x, augment=False, profile=False, visualize=False, **kwargs):
        # During super().__init__(), ultralytics calls forward() with a 3-ch
        # tensor to compute strides — range_encoder doesn't exist yet, skip it
        if not hasattr(self, "range_encoder"):
            if isinstance(x, dict):
                return self.criterion(self._run_once(x["img"][:, :3]), x)
            return self._run_once(x[:, :3])

        # Normal training call: ultralytics passes full batch dict via model(batch)
        if isinstance(x, dict):
            preds = self._extract_and_forward(x["img"])
            return self.criterion(preds, x)

        # Inference / validation call: plain tensor
        return self._extract_and_forward(x)

    def _predict_once(self, x, profile=False, visualize=False, embed=None, **kwargs):
        """Val/predict path."""
        if not hasattr(self, "range_encoder"):
            return self._run_once(x[:, :3])
        return self._extract_and_forward(x)

    def loss(self, batch, preds=None):
        """AMP / DDP path: unwrap_model(model).loss(batch, preds)."""
        if preds is None:
            preds = self._extract_and_forward(batch["img"])
        return self.criterion(preds, batch)


# ════════════════════════════════════════════════════════════
# 3.  Dataset
# ════════════════════════════════════════════════════════════

class DualStreamDataset(YOLODataset):
    """
    Overrides load_image() to stack Comb4 (3-ch) + Range (1-ch) → 4-ch array.
    All label/instance/transform handling is inherited unchanged.
    """

    def __init__(self, *args, range_dir: str, **kwargs):
        self.range_dir = Path(range_dir)
        super().__init__(*args, **kwargs)

    def load_image(self, i):
        """Return (img_HWC, (orig_h, orig_w), (resized_h, resized_w))."""
        img_path  = Path(self.im_files[i])

        # Comb4 (3-ch BGR)
        img_comb4 = cv2.imread(str(img_path))
        if img_comb4 is None:
            raise FileNotFoundError(f"Comb4 image not found: {img_path}")

        # Range (1-ch)
        range_path = self.range_dir / img_path.name
        try:
            img_range = cv2.imread(str(range_path), cv2.IMREAD_UNCHANGED)
            if img_range is None:
                raise FileNotFoundError
            if img_range.ndim == 3:
                img_range = img_range[:, :, 0]
            img_range = img_range.astype(np.float32) / 255.0
        except Exception:
            LOGGER.warning(f"Range image not found: {range_path} — using zeros")
            img_range = np.zeros(img_comb4.shape[:2], dtype=np.uint8)
        img_4ch = np.concatenate(
            [img_comb4, img_range[:, :, None]], axis=-1
        ).astype(np.uint8)

        orig_h, orig_w = img_4ch.shape[:2]
        r = self.imgsz / max(orig_h, orig_w)
        if r != 1:
            interp = cv2.INTER_LINEAR if r > 1 else cv2.INTER_AREA
            img_4ch = cv2.resize(
                img_4ch,
                (int(orig_w * r), int(orig_h * r)),
                interpolation=interp,
            )

        return img_4ch, (orig_h, orig_w), img_4ch.shape[:2]

    @staticmethod
    def collate_fn(batch):
        # Use ultralytics default collate for all keys except img
        from ultralytics.data.dataset import YOLODataset as _YDS
        new_batch = _YDS.collate_fn(batch)
        return new_batch


# ════════════════════════════════════════════════════════════
# 4.  Trainer
# ════════════════════════════════════════════════════════════

class DualStreamTrainer(DetectionTrainer):

    def __init__(self, range_train_dir: str, range_val_dir: str, **kwargs):
        self.range_train_dir = range_train_dir
        self.range_val_dir   = range_val_dir
        super().__init__(**kwargs)
    

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DualStreamModel(
            cfg=cfg or self.args.model,
            ch=3,
            nc=self.data["nc"],
            verbose=verbose,
        )
        if weights:
            model.load(weights)
        return model

    def set_model_attributes(self):
        """Set standard attributes then ensure criterion is initialised."""
        super().set_model_attributes()
        # super() calls model.init_criterion() internally in newer ultralytics;
        # force it here for older versions that don't
        if not hasattr(self.model, "criterion") or self.model.criterion is None:
            self.model.criterion = self.model.init_criterion()

    def build_dataset(self, img_path, mode="train", batch=None):
        gs        = max(int(de_parallel(self.model).stride.max() if self.model else 0), 32)
        range_dir = self.range_train_dir if mode == "train" else self.range_val_dir
        # Mosaic is incompatible with custom load_image — force off
        self.args.mosaic = 0.0
        self.args.mixup  = 0.0
        return DualStreamDataset(
            img_path=img_path,
            imgsz=self.args.imgsz,
            batch_size=batch,
            augment=mode == "train",
            hyp=self.args,
            rect=self.args.rect,
            cache=self.args.cache or None,
            single_cls=self.args.single_cls or False,
            stride=int(gs),
            pad=0.0 if mode == "train" else 0.5,
            prefix=f"{mode}: ",
            task=self.args.task,
            classes=self.args.classes,
            data=self.data,
            fraction=self.args.fraction if mode == "train" else 1.0,
            range_dir=range_dir,
        )

In [2]:
import torch

def on_train_epoch_end(trainer):
    if trainer.epoch % 10 == 0:
        # Check range_proj gradient — confirms range features are flowing
        for name, param in trainer.model.range_proj.named_parameters():
            if param.grad is not None:
                print(f"[Epoch {trainer.epoch}] range_proj.{name} grad: {param.grad.norm().item():.4f}")

        # Check range_encoder gradient — confirms encoder is actually learning
        for name, param in trainer.model.range_encoder.named_parameters():
            if param.grad is not None:
                print(f"[Epoch {trainer.epoch}] range_encoder.{name} grad: {param.grad.norm().item():.4f}")

        # Check if range_encoder weights are changing over time
        # (save first layer weight norm as a proxy)
        first_conv_norm = trainer.model.range_encoder.encode[0].conv.weight.norm().item()
        print(f"[Epoch {trainer.epoch}] range_encoder first conv weight norm: {first_conv_norm:.4f}")

In [3]:
RANGE_TRAIN = "dataset_seperate_range/images_range/train"
RANGE_VAL   = "dataset_seperate_range/images_range/valid"

trainer = DualStreamTrainer(
    range_train_dir=RANGE_TRAIN,
    range_val_dir=RANGE_VAL,
    overrides=dict(
    model    = "yolo11n.yaml",
    data     = "dual_stream.yaml",
    imgsz    = 1024,
    epochs   = 500,
    patience = 80,
    batch    = 8,
    device   = 0,
    project  = "dual_stream",
    name     = "yolo11n_comb4_range",
    amp      = False,
    augment  = False,
    mosaic   = 0.0,
    mixup    = 0.0,
    workers  = 0,
),
)
def on_val_start(trainer):
    print("\n[VALIDATION STARTED — checking channels]")

trainer.add_callback("on_val_start", on_val_start)
trainer.add_callback("on_train_epoch_end", on_train_epoch_end)
trainer.train()

Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dual_stream.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolo11n_comb4_range31, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=80, p

In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\To be shifted\Snowpole Detection\dual_stream\yolo11n_comb4_range\results.csv")
df.columns = df.columns.str.strip()

best = df.loc[df["metrics/mAP50(B)"].idxmax()]
print(f"Best epoch   : {int(best['epoch'])}")
print(f"mAP50        : {best['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95     : {best['metrics/mAP50-95(B)']:.4f}")
print(f"Precision    : {best['metrics/precision(B)']:.4f}")
print(f"Recall       : {best['metrics/recall(B)']:.4f}")

In [5]:
batch = next(iter(trainer.train_loader))
print(batch["img"].shape)
print(batch["img"][:, 3].float().mean())
print(batch["img"][:, 3].float().min(), batch["img"][:, 3].float().max())

torch.Size([8, 4, 1024, 1024])
tensor(92.8991)
tensor(0.) tensor(255.)


In [9]:
import pandas as pd

df = pd.read_csv(r"dual_stream\yolo11n_comb4_range27\results.csv")
df.columns = df.columns.str.strip()

best = df.loc[df["metrics/mAP50(B)"].idxmax()]
print(f"Best epoch   : {int(best['epoch'])}")
print(f"mAP50        : {best['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95     : {best['metrics/mAP50-95(B)']:.4f}")
print(f"Precision    : {best['metrics/precision(B)']:.4f}")
print(f"Recall       : {best['metrics/recall(B)']:.4f}")

Best epoch   : 235
mAP50        : 0.9123
mAP50-95     : 0.4293
Precision    : 0.9114
Recall       : 0.8745


In [22]:
RANGE_TEST = r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_range\test"

trainer = DualStreamTrainer(
    range_train_dir=RANGE_TEST,
    range_val_dir=RANGE_TEST,
    overrides=dict(
        model    = r"C:\To be shifted\Snowpole Detection\dual_stream\yolo11n_comb4_range27\weights\best.pt",
        data     = "dual_stream_test.yaml",
        imgsz    = 1024,
        batch    = 8,
        device   = 0,
        workers  = 0,
        epochs   = 1,
        patience = 0,
        exist_ok = True,
        mosaic   = 0.0,
        mixup    = 0.0,
    ),
)
trainer.train()

Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dual_stream_test.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\To be shifted\Snowpole Detection\dual_stream\yolo11n_comb4_range27\weights\best.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimiz

TypeError: DualStreamModel.forward() got an unexpected keyword argument 'embed'

In [19]:
from ultralytics.utils import DEFAULT_CFG
# from dual_stream_model import DualStreamDataset
from pathlib import Path

ds = DualStreamDataset(
    img_path=r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images\test",
    imgsz=1024,
    batch_size=8,
    augment=False,
    hyp=DEFAULT_CFG,        # ← fix
    rect=False,
    cache=False,
    single_cls=False,
    stride=32,
    pad=0.5,
    prefix="test: ",
    task="detect",
    classes=None,
    data={"nc": 1, "names": {0: "snowpole"}, "channels": 3},
    fraction=1.0,
    range_dir=r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_range\test",
)

for i in range(3):
    img_path = Path(ds.im_files[i])
    range_path = ds.range_dir / img_path.name
    print(f"Comb4 : {img_path}")
    print(f"Range : {range_path}")
    print(f"Range exists: {range_path.exists()}")
    print()

test: Fast image access  (ping: 0.00.0 ms, read: 1028.71058.0 MB/s, size: 585.8 KB)
test: Scanning C:\To be shifted\Snowpole Detection\dataset_seperate_range\labels\test.cache... 197 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 197/197  0.0s
Comb4 : C:\To be shifted\Snowpole Detection\dataset_seperate_range\images\test\image_0.png
Range : C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_range\test\image_0.png
Range exists: True

Comb4 : C:\To be shifted\Snowpole Detection\dataset_seperate_range\images\test\image_1009.png
Range : C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_range\test\image_1009.png
Range exists: True

Comb4 : C:\To be shifted\Snowpole Detection\dataset_seperate_range\images\test\image_1017.png
Range : C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_range\test\image_1017.png
Range exists: True



In [17]:
from pathlib import Path

test_range = Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_range\test")
test_comb4 = Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images\test")

comb4_names = {p.name for p in test_comb4.glob("*.png")}
range_names = {p.name for p in test_range.glob("*.png")}

missing = comb4_names - range_names
print(f"Missing from range: {len(missing)}")
print(f"Examples: {list(missing)[:5]}")

Missing from range: 0
Examples: []


In [ ]:
# Run this after trainer.train() starts, or hook it into a callback
import torch

# Check gate value — tells us if range branch is opening up
gate_val = torch.sigmoid(trainer.model.range_encoder.gate).item()
print(f"Range gate: {gate_val:.4f}  (0.002=silent, 0.5=full, anything >0.01 = learning)")

# Check if range_proj weights are getting gradients
for name, param in trainer.model.range_proj.named_parameters():
    if param.grad is not None:
        print(f"range_proj.{name} grad norm: {param.grad.norm().item():.6f}")
    else:
        print(f"range_proj.{name}: NO GRADIENT")

In [ ]:
from pathlib import Path

label_dir = Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\labels\train")
img_dir   = Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train")

imgs   = list(img_dir.glob("*.png"))
labels = list(label_dir.glob("*.txt"))

print(f"Images : {len(imgs)}")
print(f"Labels : {len(labels)}")

# Check if stems match
img_stems = {p.stem for p in imgs}
lbl_stems = {p.stem for p in labels}

print(f"Matched: {len(img_stems & lbl_stems)}")
print(f"Images without labels: {len(img_stems - lbl_stems)}")
print(f"Sample image name : {imgs[0].name}")
print(f"Sample label name : {labels[0].name}")

Images : 1367
Labels : 1367
Matched: 1367
Images without labels: 0
Sample image name : image_1.png
Sample label name : image_1.txt


In [ ]:
# from pathlib import Path

# # Delete stale cache files — they locked in the wrong label path
# for cache in Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4").rglob("*.cache"):
#     cache.unlink()
#     print(f"Deleted: {cache}")

Deleted: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train.cache
Deleted: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\valid.cache
Deleted: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\labels\train.cache
Deleted: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\labels\valid.cache


In [ ]:
import shutil
from pathlib import Path

src_labels = Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\labels")
# Ultralytics will look here: replace 'images_comb4' with 'labels' in the train path
# train path = .../images_comb4/train  → label path = .../labels/train  ✓
# That's actually correct! So let's check what ultralytics is actually resolving to

from ultralytics.data.utils import img2label_paths
import glob

imgs = glob.glob(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train\*.png")
lbls = img2label_paths(imgs)
print("First image:", imgs[0])
print("Derived label:", lbls[0])

First image: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train\image_1.png
Derived label: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train\image_1.txt


In [ ]:
from ultralytics.data.utils import check_det_dataset
print(check_det_dataset("dual_stream.yaml"))

{'path': WindowsPath('C:/To be shifted/Snowpole Detection/dataset_seperate_range'), 'nc': 1, 'train': 'C:\\To be shifted\\Snowpole Detection\\dataset_seperate_range\\images_comb4\\train', 'val': 'C:\\To be shifted\\Snowpole Detection\\dataset_seperate_range\\images_comb4\\valid', 'test': 'C:\\To be shifted\\Snowpole Detection\\dataset_seperate_range\\images_comb4\\test', 'names': {0: 'snowpole'}, 'yaml_file': 'dual_stream.yaml', 'channels': 3}


In [1]:
from pathlib import Path

for p in Path(r"C:\To be shifted\Snowpole Detection").rglob("*.cache"):
    print("Deleting:", p)
    p.unlink()

Deleting: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train.cache
Deleting: C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\valid.cache


In [2]:
from pathlib import Path

img_path = Path(r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train\image_1.png")
label_guess = Path(str(img_path).replace("images", "labels")).with_suffix(".txt")

print(label_guess)
print("Exists:", label_guess.exists())

C:\To be shifted\Snowpole Detection\dataset_seperate_range\labels_comb4\train\image_1.txt
Exists: True


In [3]:
import os

img_dir = r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\images_comb4\train"
lbl_dir = r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\labels_comb4\train"

missing = []
for f in os.listdir(img_dir):
    if f.endswith((".png",".jpg",".jpeg")):
        label = os.path.splitext(f)[0] + ".txt"
        if not os.path.exists(os.path.join(lbl_dir, label)):
            missing.append(label)

print("Missing labels:", len(missing))
print(missing[:10])

Missing labels: 0
[]


In [4]:
import os

lbl_dir = r"C:\To be shifted\Snowpole Detection\dataset_seperate_range\labels_comb4\train"

bad = []
for f in os.listdir(lbl_dir):
    if f.endswith(".txt"):
        with open(os.path.join(lbl_dir, f)) as file:
            for line in file:
                if line.strip():
                    cls = int(line.split()[0])
                    if cls != 0:
                        bad.append((f, cls))

print("Bad labels:", bad[:10])

Bad labels: []


In [5]:
bad_vals = []
for f in os.listdir(lbl_dir):
    if f.endswith(".txt"):
        with open(os.path.join(lbl_dir, f)) as file:
            for line in file:
                parts = list(map(float, line.split()))
                if any(v > 1 for v in parts[1:]):
                    bad_vals.append(f)

print("Non-normalized:", bad_vals[:10])

Non-normalized: []


In [4]:
gate_grad = trainer.model.range_encoder.gate.grad
print(f"Gate grad: {gate_grad}")

Gate grad: tensor([-0.0018], device='cuda:0')
